# Yelp Business-Level Prediction Template

This template builds a reusable business-level modeling table for predicting `average_future_rating`.

It is designed to be reused by scenario notebooks such as 30D1Y_PRECOVID and 30D1Y_COVID.

Entry points: call `build_prediction_dataset(...)` to build features, `run_sanity_checks(...)` to validate the table, and `run_prediction_pipeline(...)` to run everything end to end.

Next step: the regression modeling block uses the prepared business-level table to fit and compare models.

Workflow:
1. Data filtration: load review-level features, merge dates, and define cohort/window labels.
2. Data processing: aggregate early-window reviews into business-level features.
3. Modeling table: build the final business-level dataset with the future target.
4. Sanity checks: validate the final table before modeling.
5. Regression modeling: train and score baseline and regression models.
6. Entry point: call `run_prediction_pipeline(...)` when you want the full template workflow.

## Data Filtration Block

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split

from xgboost import XGBRegressor


In [11]:
def load_review_feature_table(processed_dir: str | Path, feature_file: str = "review_features.csv") -> pd.DataFrame:
    """Load the final review feature table (one row per review_id)."""
    processed_dir = Path(processed_dir)
    feature_path = processed_dir / feature_file

    if not feature_path.exists():
        raise FileNotFoundError(f"Could not find {feature_path.name} in {processed_dir}")

    df = pd.read_csv(feature_path)

    required = ["review_id", "business_id", "stars", "date"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Feature file missing required columns: {missing}")

    # Enforce datetime consistency (same as original pipeline)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # Enforce one row per review_id (matches validate="1:1" intent)
    if df.duplicated(subset=["review_id"]).any():
        raise ValueError("Duplicate review_id values found; expected one row per review.")

    return df.copy()

In [12]:
def assign_cohort_and_windows(
    review_features_df: pd.DataFrame,
    cohort_year: int,
    early_window_days: int,
    future_window_end_day: int,
    period: str | None = None,
) -> pd.DataFrame:
    """Assign t0, day offsets, and early/future labels at review level."""
    out = review_features_df.copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")

    first_review_df = (
        out.groupby("business_id", as_index=False)["date"]
        .min()
        .rename(columns={"date": "first_review_date"})
    )

    cohort_business_df = first_review_df[first_review_df["first_review_date"].dt.year == cohort_year].copy()

    if period is not None:
        period_upper = period.upper()
        if period_upper == "PRECOVID":
            cohort_business_df = cohort_business_df[cohort_business_df["first_review_date"] < pd.Timestamp("2020-03-01")]
        elif period_upper == "COVID":
            cohort_business_df = cohort_business_df[cohort_business_df["first_review_date"] >= pd.Timestamp("2020-03-01")]
        else:
            raise ValueError("period must be one of: None, 'PRECOVID', 'COVID'")

    out = out[out["business_id"].isin(cohort_business_df["business_id"])].copy()
    out = out.merge(cohort_business_df, on="business_id", how="inner", validate="m:1")

    out["review_day_offset"] = (out["date"] - out["first_review_date"]).dt.days
    out["is_early"] = out["review_day_offset"].between(0, early_window_days, inclusive="both")
    out["is_future"] = out["review_day_offset"].between(early_window_days + 1, future_window_end_day, inclusive="both")

    return out.copy()

## Feature Dictionary

### Target
- `average_future_rating`: Mean `stars` in the future window at business level.

### Expanded Early-Window Features From features_extraction.ipynb
- All numeric review-level features are aggregated to business-level means within the early window.
- The business-level feature set includes `early_review_count` plus mean summaries for review score, vote, user profile, user behavior/time, and text metadata signals.

## Final Business-Level Features

These are the final business-level input features created by the early-window aggregation step:

- `early_review_count`
- `early_avg_stars`
- `early_avg_useful`
- `early_avg_funny`
- `early_avg_cool`
- `early_avg_review_count`
- `early_avg_fans`
- `early_avg_friends_count`
- `early_avg_compliment_writer`
- `early_avg_reviewer_bias`
- `early_avg_user_tenure_days`
- `early_avg_elite_persistence`
- `early_avg_review_word_count`
- `early_avg_review_frequency`

`business_id` is kept as the join key, and `average_future_rating` is the target.


## Data Processing Block

In [13]:
def build_early_business_features(labeled_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate all early-window review-level features to one row per business."""
    early_df = labeled_df[labeled_df["is_early"]].copy()

    id_cols = {"review_id", "user_id", "business_id", "review_day_offset", "is_early", "is_future"}
    numeric_cols = [
        col
        for col in early_df.select_dtypes(include="number").columns
        if col not in id_cols
    ]

    agg_map = {col: "mean" for col in numeric_cols}
    agg_map["review_id"] = "count"

    business_agg = early_df.groupby("business_id", as_index=False).agg(agg_map)

    rename_map = {
        "review_id": "early_review_count",
    }
    rename_map.update({f"{col}": f"early_avg_{col}" for col in numeric_cols})
    business_agg = business_agg.rename(columns=rename_map)

    return business_agg.copy()

## Modeling Table Block

In [14]:
def build_future_target(labeled_df: pd.DataFrame) -> pd.DataFrame:
    """Create future-window target table at business level."""
    future_df = labeled_df[labeled_df["is_future"]].copy()

    target_df = future_df.groupby("business_id", as_index=False).agg(
        average_future_rating=("stars", "mean"),
        future_review_count=("review_id", "count"),
    )

    return target_df.copy()


def build_modeling_table(
    labeled_df: pd.DataFrame,
    min_early_reviews: int = 2,
    min_future_reviews: int = 3,
) -> pd.DataFrame:
    """Join early-window features with future target and apply threshold filters."""
    early_features = build_early_business_features(labeled_df=labeled_df)
    future_target = build_future_target(labeled_df=labeled_df)

    model_df = early_features.merge(
        future_target,
        on="business_id",
        how="inner",
        validate="1:1",
    )

    model_df = model_df[
        (model_df["early_review_count"] >= min_early_reviews)
        & (model_df["future_review_count"] >= min_future_reviews)
    ].copy()

    model_df = model_df.drop(columns=["future_review_count"])

    return model_df.copy()


def get_default_feature_columns(model_df: pd.DataFrame) -> list[str]:
    """Return all default feature columns except business_id and target columns."""
    excluded = {"business_id", "average_future_rating"}
    return [c for c in model_df.columns if c not in excluded]

## Features Block

In [15]:
def build_prediction_dataset(
    processed_dir: str | Path,
    cohort_year: int,
    early_window_days: int,
    future_window_end_day: int,
    period: str | None = None,
    min_early_reviews: int = 2,
    min_future_reviews: int = 3,
    feature_file: str = "review_features.csv",
) -> tuple[pd.DataFrame, list[str], str]:
    """End-to-end builder returning model_df, feature columns, and target column."""
    review_feature_df = load_review_feature_table(processed_dir=processed_dir, feature_file=feature_file)

    labeled_df = assign_cohort_and_windows(
        review_features_df=review_feature_df,
        cohort_year=cohort_year,
        early_window_days=early_window_days,
        future_window_end_day=future_window_end_day,
        period=period,
    )

    model_df = build_modeling_table(
        labeled_df=labeled_df,
        min_early_reviews=min_early_reviews,
        min_future_reviews=min_future_reviews,
    )

    feature_cols = get_default_feature_columns(model_df)
    target_col = "average_future_rating"

    return model_df, feature_cols, target_col

## Sanity Check Block

This block validates the final business-level modeling table before regression modeling.


In [16]:
def run_sanity_checks(
    model_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    key_col: str = "business_id",
    strict: bool = True,
) -> dict[str, object]:
    """Run basic data-quality checks before regression modeling."""
    missing_features = [col for col in feature_cols if col not in model_df.columns]
    duplicate_keys = int(model_df.duplicated(subset=[key_col]).sum()) if key_col in model_df.columns else None
    missing_target = int(model_df[target_col].isna().sum()) if target_col in model_df.columns else None
    missing_feature_values = int(model_df[feature_cols].isna().sum().sum()) if feature_cols else 0

    report = {
        "n_rows": int(model_df.shape[0]),
        "n_columns": int(model_df.shape[1]),
        "n_features": int(len(feature_cols)),
        "missing_features": missing_features,
        "duplicate_keys": duplicate_keys,
        "missing_target": missing_target,
        "missing_feature_values": missing_feature_values,
        "target_summary": model_df[target_col].describe().to_dict() if target_col in model_df.columns else {},
    }

    print("Sanity check report")
    print(f"Rows: {report['n_rows']}")
    print(f"Columns: {report['n_columns']}")
    print(f"Features: {report['n_features']}")
    print(f"Missing features: {len(missing_features)}")
    print(f"Duplicate {key_col}: {duplicate_keys}")
    print(f"Missing target values: {missing_target}")
    print(f"Missing feature values: {missing_feature_values}")
    if report["target_summary"]:
        print("Target summary:")
        print(pd.Series(report["target_summary"]))

    if strict:
        if missing_features:
            raise ValueError(f"Missing feature columns: {missing_features}")
        if duplicate_keys and duplicate_keys > 0:
            raise ValueError(f"Found {duplicate_keys} duplicate {key_col} values")
        if missing_target and missing_target > 0:
            raise ValueError(f"Found {missing_target} missing values in {target_col}")

    return report


## Regression Modeling Block

This block trains and evaluates regression models using R^2 and MAE.


In [ ]:
def build_default_regression_models(random_state: int = 42) -> dict[str, object]:
    """Return the default regression models used by the template."""
    models: dict[str, object] = {
        "baseline_mean": DummyRegressor(strategy="mean"),
        "linear_regression": LinearRegression(),
        "ridge": Ridge(alpha=1.0),
        "random_forest": RandomForestRegressor(
            n_estimators=300,
            random_state=random_state,
            n_jobs=-1,
        ),
    }

    models["xgboost"] = XGBRegressor(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=random_state,
        n_jobs=-1,
    )

    return models


def _compute_cv_metrics(
    model: object,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    cv_folds: int,
    random_state: int,
    include_cv: bool,
) -> dict[str, float]:
    """Compute K-fold CV MAE/R^2 for a model on training data."""
    metrics = {
        "cv_mae_mean": np.nan,
        "cv_mae_std": np.nan,
        "cv_r2_mean": np.nan,
        "cv_r2_std": np.nan,
        "cv_folds_used": 0,
    }

    if not include_cv:
        return metrics

    n_train = int(X_train.shape[0])
    folds = min(int(cv_folds), n_train)
    if folds < 2:
        return metrics

    cv = KFold(n_splits=folds, shuffle=True, random_state=random_state)
    cv_scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring={"mae": "neg_mean_absolute_error", "r2": "r2"},
        n_jobs=-1,
        error_score="raise",
    )

    metrics["cv_mae_mean"] = float(-cv_scores["test_mae"].mean())
    metrics["cv_mae_std"] = float(cv_scores["test_mae"].std())
    metrics["cv_r2_mean"] = float(cv_scores["test_r2"].mean())
    metrics["cv_r2_std"] = float(cv_scores["test_r2"].std())
    metrics["cv_folds_used"] = int(folds)
    return metrics


def run_regression_modeling(
    model_df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    test_size: float = 0.2,
    random_state: int = 42,
    models: dict[str, object] | None = None,
    include_cv: bool = True,
    cv_folds: int = 5,
) -> dict[str, object]:
    """Train and score regression models with test and CV metrics."""
    working_df = model_df.dropna(subset=feature_cols + [target_col]).copy()

    # Guard: empty dataset after filtering/NA drop
    if working_df.shape[0] == 0:
        raise ValueError(
            "Model dataset is empty after filtering. "
            "Check cohort_year/period/windows and min review thresholds, "
            "and ensure processed data covers the selected time period."
        )

    non_numeric_cols = [col for col in feature_cols if not pd.api.types.is_numeric_dtype(working_df[col])]
    if non_numeric_cols:
        raise TypeError(f"Non-numeric feature columns are not supported: {non_numeric_cols}")

    X = working_df.loc[:, feature_cols]
    y = working_df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
    )

    model_specs = models or build_default_regression_models(random_state=random_state)
    results: list[dict[str, object]] = []
    predictions_df = pd.DataFrame(index=y_test.index)
    predictions_df["business_id"] = working_df.loc[y_test.index, "business_id"].values
    predictions_df[target_col] = y_test.values

    baseline_model = model_specs.get("baseline_mean")
    if baseline_model is None:
        baseline_model = DummyRegressor(strategy="mean")

    baseline_model.fit(X_train, y_train)
    baseline_pred = baseline_model.predict(X_test)
    predictions_df["pred_baseline_mean"] = baseline_pred
    baseline_cv = _compute_cv_metrics(
        model=baseline_model,
        X_train=X_train,
        y_train=y_train,
        cv_folds=cv_folds,
        random_state=random_state,
        include_cv=include_cv,
    )
    results.append(
        {
            "model": "baseline_mean",
            "mae": float(mean_absolute_error(y_test, baseline_pred)),
            "rmse": float(np.sqrt(mean_squared_error(y_test, baseline_pred))),
            "r2": float(r2_score(y_test, baseline_pred)),
            "n_train": int(X_train.shape[0]),
            "n_test": int(X_test.shape[0]),
            "n_features": int(X_train.shape[1]),
            **baseline_cv,
        }
    )

    fitted_models: dict[str, object] = {"baseline_mean": baseline_model}
    for model_name, model in model_specs.items():
        if model_name == "baseline_mean":
            continue

        model.fit(X_train, y_train)
        model_pred = model.predict(X_test)
        predictions_df[f"pred_{model_name}"] = model_pred
        cv_metrics = _compute_cv_metrics(
            model=model,
            X_train=X_train,
            y_train=y_train,
            cv_folds=cv_folds,
            random_state=random_state,
            include_cv=include_cv,
        )
        results.append(
            {
                "model": model_name,
                "mae": float(mean_absolute_error(y_test, model_pred)),
                "rmse": float(np.sqrt(mean_squared_error(y_test, model_pred))),
                "r2": float(r2_score(y_test, model_pred)),
                "n_train": int(X_train.shape[0]),
                "n_test": int(X_test.shape[0]),
                "n_features": int(X_train.shape[1]),
                **cv_metrics,
            }
        )
        fitted_models[model_name] = model

    results_df = pd.DataFrame(results).sort_values(
        ["mae", "cv_mae_mean", "r2"],
        ascending=[True, True, False],
        na_position="last",
    ).reset_index(drop=True)

    print("Regression results")
    print(results_df)

    return {
        "results_df": results_df,
        "predictions_df": predictions_df,
        "fitted_models": fitted_models,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
    }


def run_prediction_pipeline(
    processed_dir: str | Path,
    cohort_year: int,
    early_window_days: int,
    future_window_end_day: int,
    period: str | None = None,
    min_early_reviews: int = 2,
    min_future_reviews: int = 3,
    feature_file: str = "review_features.csv",
    sanity_strict: bool = True,
    test_size: float = 0.2,
    random_state: int = 42,
    models: dict[str, object] | None = None,
    include_cv: bool = True,
    cv_folds: int = 5,
) -> dict[str, object]:
    """Run the full template: build data, sanity-check it, and evaluate regressors."""
    model_df, feature_cols, target_col = build_prediction_dataset(
        processed_dir=processed_dir,
        cohort_year=cohort_year,
        early_window_days=early_window_days,
        future_window_end_day=future_window_end_day,
        period=period,
        min_early_reviews=min_early_reviews,
        min_future_reviews=min_future_reviews,
        feature_file=feature_file,
    )

    # Guard: empty modeling table before sanity checks/regression
    if model_df.shape[0] == 0:
        msg = (
            "Model dataset is empty (0 rows). This often happens when "
            "`cohort_year` and `period` conflict (e.g., COVID with a pre-2020 cohort), "
            "or thresholds/windows are too strict. For COVID use cohort_year >= 2020; "
            "for PRECOVID use cohort_year <= 2019. Also consider lowering min_early_reviews/min_future_reviews."
        )
        raise ValueError(msg)

    sanity_report = run_sanity_checks(
        model_df=model_df,
        feature_cols=feature_cols,
        target_col=target_col,
        strict=sanity_strict,
    )
    regression_report = run_regression_modeling(
        model_df=model_df,
        feature_cols=feature_cols,
        target_col=target_col,
        test_size=test_size,
        random_state=random_state,
        models=models,
        include_cv=include_cv,
        cv_folds=cv_folds,
    )

    return {
        "model_df": model_df,
        "feature_cols": feature_cols,
        "target_col": target_col,
        "sanity_report": sanity_report,
        "regression_report": regression_report,
    }

## Result Tables and Graphs Block

This block centralizes result generation so scenario notebooks can produce consistent tables and plots after running `run_prediction_pipeline(...)`.

In [21]:
def summarize_model_dataset(model_df: pd.DataFrame) -> dict[str, object]:
    """Return reusable summary tables for the modeling dataset."""
    numeric_cols = model_df.select_dtypes(include="number").columns.tolist()
    missing_values = model_df.isna().sum().rename("missing_count")
    summary_stats = model_df.describe(include="all")
    top_rows = model_df.head()

    return {
        "shape": model_df.shape,
        "columns": model_df.columns.tolist(),
        "numeric_columns": numeric_cols,
        "summary_stats": summary_stats,
        "missing_values": missing_values,
        "head": top_rows,
    }


def get_top_target_correlations(
    model_df: pd.DataFrame,
    target_col: str = "average_future_rating",
    top_n: int = 15,
    min_periods: int = 1,
    use_abs: bool = True,
) -> pd.DataFrame:
    """Return top feature correlations with the target."""
    numeric_cols = model_df.select_dtypes(include="number").columns.tolist()
    if target_col not in numeric_cols:
        return pd.DataFrame(columns=["feature", "corr"])

    corr_series = model_df[numeric_cols].corr(min_periods=min_periods)[target_col].drop(target_col, errors="ignore")
    ranked = corr_series.abs().sort_values(ascending=False) if use_abs else corr_series.sort_values(ascending=False)
    top = ranked.head(top_n)

    return pd.DataFrame({
        "feature": top.index,
        "corr": corr_series.loc[top.index].values,
    })


def plot_target_distribution(
    model_df: pd.DataFrame,
    target_col: str = "average_future_rating",
    bins: int = 30,
    figsize: tuple[int, int] = (8, 5),
) -> None:
    """Plot target distribution histogram."""
    if target_col not in model_df.columns:
        print(f"{target_col} not found; skipping target distribution plot.")
        return

    plt.figure(figsize=figsize)
    plt.hist(model_df[target_col], bins=bins, edgecolor="black")
    plt.xlabel("Average Future Rating")
    plt.ylabel("Number of Businesses")
    plt.title("Distribution of Average Future Rating")
    plt.show()


def plot_early_vs_future(
    model_df: pd.DataFrame,
    early_col: str = "early_avg_stars",
    target_col: str = "average_future_rating",
    figsize: tuple[int, int] = (8, 5),
) -> None:
    """Plot early-window average stars against future target."""
    needed = [early_col, target_col]
    if any(col not in model_df.columns for col in needed):
        print(f"Missing one of columns {needed}; skipping early-vs-future scatter.")
        return

    plt.figure(figsize=figsize)
    plt.scatter(model_df[early_col], model_df[target_col], alpha=0.4)
    plt.xlabel("Early Average Stars")
    plt.ylabel("Average Future Rating")
    plt.title("Early Average Stars vs Average Future Rating")
    plt.show()


def plot_actual_vs_predicted_best_model(
    regression_report: dict[str, object],
    target_col: str = "average_future_rating",
    figsize: tuple[int, int] = (8, 5),
) -> dict[str, object] | None:
    """Plot actual vs predicted values for the best model by MAE, then R^2."""
    results_df = regression_report.get("results_df")
    predictions_df = regression_report.get("predictions_df")

    if results_df is None or predictions_df is None or len(results_df) == 0:
        print("Regression outputs unavailable; skipping actual-vs-predicted plot.")
        return None

    ranked = results_df.sort_values(["mae", "r2"], ascending=[True, False])
    best_row = ranked.iloc[0]
    best_model = best_row["model"]
    pred_col = f"pred_{best_model}" if best_model != "baseline_mean" else "pred_baseline_mean"

    if pred_col not in predictions_df.columns or target_col not in predictions_df.columns:
        print(f"Missing {pred_col} or {target_col}; skipping actual-vs-predicted plot.")
        return None

    y_true = predictions_df[target_col].values
    y_pred = predictions_df[pred_col].values

    plt.figure(figsize=figsize)
    plt.scatter(y_true, y_pred, alpha=0.5)
    min_val = float(min(y_true.min(), y_pred.min()))
    max_val = float(max(y_true.max(), y_pred.max()))
    plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", color="red")
    plt.xlabel("Actual Average Future Rating")
    plt.ylabel(f"Predicted ({best_model})")
    plt.title("Actual vs Predicted (Best Model)")
    plt.show()

    print("Best model:", best_model)
    print(best_row)

    return {
        "best_model": best_model,
        "best_row": best_row,
        "pred_col": pred_col,
    }


def plot_random_forest_feature_importances(
    regression_report: dict[str, object],
    feature_cols: list[str],
    figsize: tuple[int, int] = (10, 6),
) -> pd.DataFrame:
    """Plot and return random forest feature importances if available."""
    fitted_models = regression_report.get("fitted_models", {})
    rf_model = fitted_models.get("random_forest")

    if rf_model is None or not hasattr(rf_model, "feature_importances_"):
        print("Random Forest model not available or missing feature_importances_.")
        return pd.DataFrame(columns=["feature", "importance"])

    importances = rf_model.feature_importances_
    order = np.argsort(importances)[::-1]
    feat = np.array(feature_cols)[order]
    imp = importances[order]

    plt.figure(figsize=figsize)
    plt.bar(range(len(imp)), imp, tick_label=feat)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Importance")
    plt.title("Random Forest Feature Importances")
    plt.tight_layout()
    plt.show()

    return pd.DataFrame({
        "feature": feat,
        "importance": imp,
    })


def generate_results_package(
    pipeline_output: dict[str, object],
    target_col: str = "average_future_rating",
    top_n_correlations: int = 15,
    show_plots: bool = True,
) -> dict[str, object]:
    """Generate reusable result tables and optional plots from pipeline output."""
    model_df = pipeline_output["model_df"]
    feature_cols = pipeline_output["feature_cols"]
    regression_report = pipeline_output["regression_report"]

    dataset_summary = summarize_model_dataset(model_df)
    correlation_table = get_top_target_correlations(
        model_df=model_df,
        target_col=target_col,
        top_n=top_n_correlations,
    )

    best_model_info = None
    rf_importance_table = pd.DataFrame(columns=["feature", "importance"])

    if show_plots:
        plot_target_distribution(model_df=model_df, target_col=target_col)
        plot_early_vs_future(model_df=model_df, target_col=target_col)
        best_model_info = plot_actual_vs_predicted_best_model(
            regression_report=regression_report,
            target_col=target_col,
        )
        rf_importance_table = plot_random_forest_feature_importances(
            regression_report=regression_report,
            feature_cols=feature_cols,
        )

    return {
        "dataset_summary": dataset_summary,
        "correlation_table": correlation_table,
        "best_model_info": best_model_info,
        "rf_importance_table": rf_importance_table,
    }

## Pipeline Entry Point and Example Execution

This example creates a dataset for a 30D1Y PRECOVID setup and runs sanity checks plus regression modeling.

From a scenario notebook inside `previous/`, import this template with: `%run ../prediction_template.ipynb`

Use `run_prediction_pipeline(...)` as the main entry point when you want the full workflow.

In [26]:
def run_example_prediction_pipeline(show_plots: bool = True) -> dict[str, object]:
    """Run the documented PRECOVID example explicitly when needed."""
    PROCESSED_DIR = Path("./processed_data")
    COHORT_YEAR = 2015
    EARLY_WINDOW_DAYS = 30
    FUTURE_WINDOW_END_DAY = 365
    PERIOD = "PRECOVID"
    MIN_EARLY_REVIEWS = 5
    MIN_FUTURE_REVIEWS = 5

    pipeline_output = run_prediction_pipeline(
        processed_dir=PROCESSED_DIR,
        cohort_year=COHORT_YEAR,
        early_window_days=EARLY_WINDOW_DAYS,
        future_window_end_day=FUTURE_WINDOW_END_DAY,
        period=PERIOD,
        min_early_reviews=MIN_EARLY_REVIEWS,
        min_future_reviews=MIN_FUTURE_REVIEWS,
        feature_file="review_features.csv",
    )

    model_df = pipeline_output["model_df"]
    feature_cols = pipeline_output["feature_cols"]
    target_col = pipeline_output["target_col"]

    print("Model dataset shape:", model_df.shape)
    print("Target column:", target_col)
    print("Number of features:", len(feature_cols))
    print("First 10 feature columns:", feature_cols[:10])
    print("Sanity report:", pipeline_output["sanity_report"])
    print("Regression summary:")
    print(pipeline_output["regression_report"]["results_df"])

    output_dir = PROCESSED_DIR / "modeling_tables"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"business_level_{EARLY_WINDOW_DAYS}D_{FUTURE_WINDOW_END_DAY}D_{PERIOD}.csv"
    model_df.to_csv(output_path, index=False, float_format="%.4f")
    print("Saved modeling table to:", output_path.resolve())

    results_package = generate_results_package(
        pipeline_output=pipeline_output,
        target_col=target_col,
        top_n_correlations=15,
        show_plots=show_plots,
    )

    return {
        "pipeline_output": pipeline_output,
        "results_package": results_package,
    }